# 10 · Model Seçimi & Hiperparametre Ayarı
1. **Baseline** kur → 2. Modelleri **aynı çapraz doğrulama katlarıyla** karşılaştır → 3. **GridSearch / RandomizedSearch** ile ayarla → 4. Test setinde **bir kez** değerlendir → 5. Kaydet.

> Test seti sadece en sonda, **bir kez** kullanılır. Model seçimi ve hiperparametre ayarı yalnızca train verisi üzerinde çapraz doğrulama ile yapılır.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
%matplotlib inline

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, recall_score,
                             classification_report, ConfusionMatrixDisplay, RocCurveDisplay)

from sklearn.model_selection import (cross_val_score, cross_validate, StratifiedKFold,
                                     GridSearchCV, RandomizedSearchCV, learning_curve)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from scipy.stats import randint, uniform
import joblib, time

In [ ]:
X, y = load_breast_cancer(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)   # tüm modeller AYNI katlar

## 1. Çapraz doğrulama ile karşılaştırma

In [ ]:
adaylar = {
    "Baseline (çoğunluk)": DummyClassifier(strategy="most_frequent"),
    "Lojistik":     Pipeline([("s", StandardScaler()), ("m", LogisticRegression(max_iter=1000))]),
    "RandomForest": RandomForestClassifier(random_state=42),
    "XGBoost":      XGBClassifier(eval_metric="logloss", random_state=42),
}
cv_sonuc = {}
for ad, m in adaylar.items():
    r = cross_validate(m, X_train, y_train, cv=cv, scoring=["f1", "roc_auc"])
    cv_sonuc[ad] = r
    print(f"{ad:20s} F1 = {r['test_f1'].mean():.3f} ± {r['test_f1'].std():.3f}   "
          f"AUC = {r['test_roc_auc'].mean():.3f}   süre = {r['fit_time'].mean():.2f}s")

In [ ]:
plt.figure(figsize=(9, 4))
sns.boxplot(data=pd.DataFrame({ad: r["test_f1"] for ad, r in cv_sonuc.items() if "Baseline" not in ad}), palette="Set2")
plt.title("5-katlı CV F1 dağılımı"); plt.ylabel("F1"); plt.show()

**Yorum:** Ortalama kadar **standart sapma** da önemlidir: kutular üst üste biniyorsa modeller arasındaki fark istatistiksel olarak anlamlı olmayabilir. Baseline, 'hiçbir şey öğrenmeyen' modelin skorunu gösterir; diğer modeller onu açıkça geçmeli. Eğitim süresi de üretim için bir seçim kriteridir.

## 2. GridSearchCV (tüm kombinasyonlar)

In [ ]:
t0 = time.time()
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid={"n_estimators": [100, 300], "max_depth": [None, 5, 10], "min_samples_leaf": [1, 3]},
    cv=cv, scoring="f1", n_jobs=-1,
).fit(X_train, y_train)
print(f"Süre: {time.time()-t0:.1f}s | Denenen kombinasyon: {len(grid.cv_results_['params'])}")
print("En iyi parametreler:", grid.best_params_)
print("En iyi CV F1:", round(grid.best_score_, 3))
pd.DataFrame(grid.cv_results_)[["params", "mean_test_score", "std_test_score", "rank_test_score"]].sort_values("rank_test_score").head()

## 3. RandomizedSearchCV (geniş arama uzayında rastgele örnekleme)

In [ ]:
rand = RandomizedSearchCV(
    XGBClassifier(eval_metric="logloss", random_state=42),
    param_distributions={"n_estimators": randint(100, 600), "max_depth": randint(2, 8),
                         "learning_rate": uniform(0.01, 0.3), "subsample": uniform(0.6, 0.4),
                         "colsample_bytree": uniform(0.6, 0.4)},
    n_iter=30, cv=cv, scoring="f1", n_jobs=-1, random_state=42,
).fit(X_train, y_train)
print("En iyi parametreler:", {k: round(v, 3) if isinstance(v, float) else v for k, v in rand.best_params_.items()})
print("En iyi CV F1:", round(rand.best_score_, 3))

**Yorum:** Grid search her kombinasyonu dener (parametre sayısıyla üstel büyür). Randomized search sabit bütçeyle (`n_iter`) geniş bir uzayı tarar; çok parametreli modellerde genellikle daha verimlidir. Daha gelişmiş seçenek: **Optuna** (Bayesian optimizasyon).

## 4. Öğrenme eğrisi – daha fazla veri işe yarar mı?

In [ ]:
boyut, tr_skor, val_skor = learning_curve(grid.best_estimator_, X_train, y_train, cv=cv, scoring="f1",
                                          train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1)
plt.figure(figsize=(8, 4))
plt.plot(boyut, tr_skor.mean(1), "o-", color="tab:blue", label="Train")
plt.plot(boyut, val_skor.mean(1), "o-", color="tab:orange", label="Validation")
plt.fill_between(boyut, val_skor.mean(1) - val_skor.std(1), val_skor.mean(1) + val_skor.std(1), color="tab:orange", alpha=0.2)
plt.title("Öğrenme Eğrisi"); plt.xlabel("Eğitim örneği sayısı"); plt.ylabel("F1"); plt.legend(); plt.show()

**Yorum:** Validation eğrisi hâlâ yükseliyorsa **daha fazla veri** faydalı olur. Train ve validation arasındaki kalıcı boşluk overfitting'e, ikisinin birlikte düşük kalması underfitting'e işaret eder.

## 5. Final: test setinde tek değerlendirme ve kaydetme

In [ ]:
final_model = grid.best_estimator_ if grid.best_score_ >= rand.best_score_ else rand.best_estimator_
p = final_model.predict(X_test)
print("Seçilen model:", type(final_model).__name__)
print(f"Test F1: {f1_score(y_test, p):.3f} | Test ROC-AUC: {roc_auc_score(y_test, final_model.predict_proba(X_test)[:, 1]):.3f}")
ConfusionMatrixDisplay.from_predictions(y_test, p, display_labels=["kötü", "iyi"], cmap="Blues"); plt.show()

joblib.dump(final_model, "final_model.joblib")
yuklenen = joblib.load("final_model.joblib")
print("Yüklenen model aynı tahmini veriyor mu?", (yuklenen.predict(X_test) == p).all())

**Yorum:** Test skoru CV skoruna yakınsa model seçim süreci güvenilirdir. Kaydedilen `.joblib` dosyası başka bir uygulamada (Flask/FastAPI) yüklenip tahmin için kullanılabilir.

## Özet
### Data Analysis Key Findings
- Tüm modeller aynı 5-katlı CV ile karşılaştırıldı; hepsi baseline'ı (%63 çoğunluk sınıfı) açıkça geçti.
- Hiperparametre araması CV skorunda küçük ama tutarlı iyileşme sağladı; test skoru CV skoruyla uyumlu.
### Insights or Next Steps
- Yorumlanabilirlik gerekiyorsa **SHAP** ile tahminleri açıklayın.
- Sırada: **11_Model_Secim_Rehberi** (hangi model ne zaman?).